<a href="https://colab.research.google.com/github/chithrakumardakshan-cloud/northstar-analytics/blob/main/notebooks/01_sql_in_r.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# Install necessary packages
install.packages(c('sqldf', 'dplyr', 'ggplot2'))

# Load libraries
library(sqldf)
library(dplyr)
library(ggplot2)

# 2. Define the path (Since they are uploaded to the sidebar, use './')
path <- "./"

# 3. Load files
# Using stringsAsFactors=FALSE is a good practice for older R versions
orders     <- read.csv(paste0(path, "orders.csv"))
deliveries <- read.csv(paste0(path, "deliveries.csv"))
customers  <- read.csv(paste0(path, "customers.csv"))
drivers    <- read.csv(paste0(path, "drivers.csv"))
vehicles   <- read.csv(paste0(path, "vehicles.csv"))
hubs       <- read.csv(paste0(path, "hubs.csv"))
incidents  <- read.csv(paste0(path, "incidents.csv"))
complaints <- read.csv(paste0(path, "complaints.csv"))
app_events <- read.csv(paste0(path, "app_events.csv"))

Installing packages into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

also installing the dependencies ‘gsubfn’, ‘proto’, ‘RSQLite’, ‘chron’


Loading required package: gsubfn

Loading required package: proto

Warning message:
“no DISPLAY variable so Tk is not available”
Loading required package: RSQLite


Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union




In [3]:
result1 <- sqldf(
  "SELECT d.hub_id,
          h.hub_name,
          h.zone,
          d.delivery_status,
          COUNT(*) AS count,
          ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (PARTITION BY d.hub_id), 2) AS pct
   FROM deliveries d
   JOIN hubs h ON d.hub_id = h.hub_id
   GROUP BY d.hub_id, h.hub_name, h.zone, d.delivery_status
   ORDER BY d.hub_id, count DESC"
)
print(result1)


   hub_id       hub_name      zone delivery_status count   pct
1     H01 North Exchange     North          OnTime    93 68.38
2     H01 North Exchange     North         Delayed    26 19.12
3     H01 North Exchange     North          Failed    17 12.50
4     H02     South Link     South          OnTime    70 66.04
5     H02     South Link     South         Delayed    26 24.53
6     H02     South Link     South          Failed    10  9.43
7     H03      East Dock      East          OnTime    85 71.43
8     H03      East Dock      East         Delayed    23 19.33
9     H03      East Dock      East          Failed    11  9.24
10    H04      West Gate      West          OnTime    83 65.35
11    H04      West Gate      West         Delayed    28 22.05
12    H04      West Gate      West          Failed    16 12.60
13    H05   Central Core   Central          OnTime    67 58.26
14    H05   Central Core   Central         Delayed    25 21.74
15    H05   Central Core   Central          Failed    2

In [4]:
result2 <- sqldf(
  "SELECT d.driver_id,
          dr.base_zone,
          dr.employment_type,
          COUNT(*) AS total_deliveries,
          SUM(d.manual_route_override_count) AS total_overrides,
          ROUND(AVG(d.manual_route_override_count), 2) AS avg_overrides,
          ROUND(AVG(d.customer_rating_post_delivery), 2) AS avg_rating
   FROM deliveries d
   JOIN drivers dr ON d.driver_id = dr.driver_id
   GROUP BY d.driver_id, dr.base_zone, dr.employment_type
   HAVING total_deliveries >= 3
   ORDER BY avg_overrides DESC
   LIMIT 20"
)
print(result2)


   driver_id base_zone employment_type total_deliveries total_overrides
1       D127   CENTRAL        FullTime                6              17
2       D062     South        FullTime                3               6
3       D069     NORTH        PartTime                7              14
4       D085     North        PartTime                4               8
5       D105 RiverSide        Contract                7              14
6       D124     north        FullTime                4               8
7       D130      WEST        FullTime                8              16
8       D139     South        FullTime                5              10
9       D028     North        FullTime                7              13
10      D027   AIRPORT        PartTime                6              11
11      D143   CENTRAL        FullTime                5               9
12      D003   AIRPORT        FullTime                4               7
13      D107 RiverSide        FullTime                4         

In [5]:
result3 <- sqldf(
  "SELECT c.customer_id,
          c.home_zone,
          c.customer_type,
          c.loyalty_score,
          COUNT(DISTINCT cp.complaint_id) AS complaint_count,
          COUNT(DISTINCT CASE WHEN d.delivery_status = 'Failed' THEN d.delivery_id END) AS failed_deliveries,
          SUM(cp.compensation_amount) AS total_compensation
   FROM customers c
   LEFT JOIN complaints cp ON c.customer_id = cp.customer_id
   LEFT JOIN orders o ON c.customer_id = o.customer_id
   LEFT JOIN deliveries d ON o.order_id = d.order_id
   GROUP BY c.customer_id, c.home_zone, c.customer_type, c.loyalty_score
   HAVING complaint_count >= 2
   ORDER BY complaint_count DESC, failed_deliveries DESC
   LIMIT 20"
)
print(result3)


   customer_id home_zone customer_type loyalty_score complaint_count
1        C0368     North      Consumer          49.5               4
2        C0110      EAST      Consumer            NA               3
3        C0282 RiverSide      Consumer          71.4               3
4        C0372      West      Consumer          26.2               3
5        C0573   AIRPORT           SME          57.3               3
6        C0142     SOUTH      Consumer          47.0               3
7        C0172     north      Consumer          75.4               3
8        C0191     North      Consumer          58.9               3
9        C0242      East      Consumer          83.8               3
10       C0421   CENTRAL      Consumer          59.0               3
11       C0545     SOUTH      Consumer          66.9               3
12       C0626     SOUTH      Consumer          61.6               3
13       C0004   CENTRAL      Consumer          32.5               2
14       C0023     South      Cons

In [6]:
result4 <- sqldf(
  "SELECT
     CASE
       WHEN v.battery_health_pct >= 80 THEN 'Good (80-100%)'
       WHEN v.battery_health_pct >= 60 THEN 'Fair (60-79%)'
       ELSE 'Poor (<60%)'
     END AS battery_band,
     COUNT(DISTINCT v.vehicle_id) AS vehicles,
     COUNT(i.incident_id) AS total_incidents,
     ROUND(AVG(i.resolved_hours), 2) AS avg_resolve_hours
   FROM vehicles v
   JOIN deliveries d ON v.vehicle_id = d.vehicle_id
   LEFT JOIN incidents i ON d.delivery_id = i.delivery_id
   GROUP BY battery_band
   ORDER BY vehicles DESC"
)
print(result4)


    battery_band vehicles total_incidents avg_resolve_hours
1  Fair (60-79%)       56             121             12.82
2 Good (80-100%)       47             115             11.41
3    Poor (<60%)       17              44             11.31


In [7]:
result5 <- sqldf(
  "SELECT o.pickup_zone,
          o.service_type,
          COUNT(o.order_id) AS order_count,
          ROUND(AVG(o.order_value), 2) AS avg_order_value,
          ROUND(AVG(d.fuel_or_charge_cost), 2) AS avg_fuel_cost,
          ROUND(AVG(o.order_value - d.fuel_or_charge_cost), 2) AS avg_net_margin,
          ROUND(AVG(d.route_distance_km), 2) AS avg_distance_km
   FROM orders o
   JOIN deliveries d ON o.order_id = d.order_id
   GROUP BY o.pickup_zone, o.service_type
   ORDER BY avg_net_margin ASC"
)
print(result5)


   pickup_zone service_type order_count avg_order_value avg_fuel_cost
1         West     Business           7           53.95         13.15
2         East      Medical           9           52.86         11.89
3          Ctr      Medical           2           53.92         11.13
4        north      Medical           7           57.49         14.17
5      AIRPORT     Business           5           66.61         18.22
6      Central    Passenger          20           61.75         11.90
7    RiverSide       Parcel          17           68.07         13.28
8        North       Parcel           8           67.15         10.84
9         East       Retail          20           69.57         11.73
10        West       Retail           8           70.02         10.40
11        EAST     Business           7           73.71         12.96
12   RiverSide    Passenger          19           73.98         13.17
13        WEST       Retail          14           74.22         11.62
14     Central      

In [9]:
# BEFORE: Unoptimised — joins full tables then filters
system.time({
 slow <- sqldf(
   "SELECT d.*, o.pickup_zone, o.service_type, o.order_value
    FROM deliveries d JOIN orders o ON d.order_id = o.order_id
    WHERE d.delivery_status = 'Failed'")
})

# AFTER: Optimised — pre-filter in R, then join smaller dataset
failed_del <- deliveries[deliveries$delivery_status == 'Failed', ]  # 132 rows
system.time({
 fast <- sqldf(
   "SELECT d.*, o.pickup_zone, o.service_type, o.order_value
    FROM failed_del d JOIN orders o ON d.order_id = o.order_id")
})

   user  system elapsed 
  0.074   0.012   0.093 

   user  system elapsed 
  0.064   0.017   0.089 